# Automotive RAG Question Answering System - Master Notebook

Welcome! This unified notebook is the primary entry point for executing the complete Automotive RAG pipeline. It is optimized for Google Colab T4 GPU environments and serves as the definitive reference for dissertation evaluation, demonstrations, and supervisor reviews.

### Pipeline Overview:
* **Phase 1:** Environment Setup, Document Upload, Extraction, Chunking, and FAISS Vector Indexing.
* **Phase 2:** RAG Execution (Retrieval, Prompting, and LLM Inference using 4-bit Quantization).
* **Phase 3:** Automated Benchmarking & Context Window Evaluation Study.
* **Phase 4:** Dissertation Analytics & Final Summaries.

## Section 1: Environment Setup
First, we detect the environment, clone the backend repository, install the dependencies (including `tesseract-ocr`), and verify the hardware accelerators.

In [ ]:
import os
import sys
import platform

print("--- System Verification ---")
print(f"Python Version: {platform.python_version()}")

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("\nGoogle Colab Environment Detected.")
    # Clone the repository if not already present
    if not os.path.exists('Automotive-RAG-QA-System'):
        print("Cloning repository...")
        !git clone https://github.com/Adityakumar001-usn/Automotive-RAG-QA-System.git
    os.chdir('Automotive-RAG-QA-System')
else:
    print("\nLocal Environment Detected.")

sys.path.append(os.getcwd())

print("\n--- Installing Dependencies (This may take a minute) ---")
!apt-get update > /dev/null 2>&1
!apt-get install -y tesseract-ocr > /dev/null 2>&1
!pip install -r requirements.txt > /dev/null 2>&1
print("Dependencies Installed Successfully!")

import torch
print("\n--- Hardware Verification ---")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Execution will be slow. Please ensure T4 GPU is selected in Colab Runtime settings.")

## Section 2: Phase 1 - Data Ingestion & Indexing
Upload your raw Automotive PDFs, CSVs, or TXT files. The system will extract the text, clean it, chunk it, and save the embeddings offline to a CPU FAISS index.

In [ ]:
from src.processing import process_document
from src.chunking import recursive_chunking
from src.embeddings import generate_embeddings
from src.vector_store import VectorStore
from IPython.display import clear_output
import pandas as pd

file_paths = []
if IN_COLAB:
    from google.colab import files
    print("Please upload your automotive documents (PDF, DOCX, TXT, CSV, XLSX, JPG, PNG):")
    uploaded = files.upload()
    os.makedirs('uploaded_docs', exist_ok=True)
    for filename, data in uploaded.items():
        path = os.path.join('uploaded_docs', filename)
        with open(path, 'wb') as f:
            f.write(data)
        file_paths.append(path)
    clear_output()
    print(f"Successfully uploaded {len(file_paths)} files.")
else:
    print("Not in Colab. Using test assets.")
    file_paths = ['test_assets/sample.pdf', 'test_assets/sample.txt']

all_chunks, all_metadatas = [], []
summary_data = []

print("\n--- Processing Documents ---")
for path in file_paths:
    file_size_kb = os.path.getsize(path) / 1024
    text, metadata = process_document(path)
    chunks = recursive_chunking(text)
    
    summary_data.append({
        'Document Name': os.path.basename(path),
        'File Type': metadata['document_type'].upper(),
        'Detected Category': metadata['category'],
        'File Size': f"{file_size_kb/1024:.2f} MB" if file_size_kb > 1024 else f"{file_size_kb:.2f} KB",
        'Chunks Generated': len(chunks)
    })
    
    all_chunks.extend(chunks)
    all_metadatas.extend([metadata.copy() for _ in chunks])

df_summary = pd.DataFrame(summary_data)
display(df_summary)

if all_chunks:
    print("\n--- Generating Embeddings & Building Index ---")
    embeddings = generate_embeddings(all_chunks)
    vs = VectorStore()
    vs.build_index(embeddings, all_chunks, all_metadatas)
    vs.save_index('faiss_index')
    
    unique_cats = set([m['category'] for m in all_metadatas])
    
    print("\n===========================")
    print("🧠 KNOWLEDGE BASE SUMMARY")
    print("===========================")
    print(f"Documents Uploaded: {len(file_paths)}")
    print(f"Categories Detected: {len(unique_cats)}")
    print(f"Chunks Generated: {len(all_chunks)}")
    print(f"Indexed Vectors: {vs.index.ntotal}")
    
    print("\n✅ Phase 1 Successful! Database built and saved to disk.")
else:
    print("\nNo chunks generated. Pipeline halted.")

## Section 3: Phase 2 - Automotive RAG Question Answering
With the database built, we initialize the `AutomotiveRAG` engine. The LLM (Phi-3) will load natively into your Colab T4 GPU using 4-bit quantization to prevent OOM errors.

In [ ]:
from src.retriever import Retriever
from src.rag_engine import AutomotiveRAG
from src.rag_evaluator import RAGEvaluator
import time

rag = AutomotiveRAG(Retriever(vs))
evaluator = RAGEvaluator()

print("\n✅ RAG Engine Loaded!")

In [ ]:
# Interactive Question Input
example_q = "What is the content of the PDF?"
question = input(f"Ask a question (or press Enter to use default: '{example_q}'): ")
if not question:
    question = example_q

start_time = time.time()
result = rag.ask(question)
total_time = time.time() - start_time

print("\n===========================")
print("🤖 AI ANSWER:")
print(result['answer'])
print(f"\n⏳ Generation Time: {total_time:.2f}s")
print("===========================")

print("\n📚 RETRIEVED SOURCES BY CATEGORY:")
grouped_sources = {}
for s in result['sources']:
    cat = s['category']
    if cat not in grouped_sources:
        grouped_sources[cat] = []
    grouped_sources[cat].append(s)

for cat, sources in grouped_sources.items():
    print(f"\n✓ {cat.replace('_', ' ').title()}")
    for s in sources:
        print(f"  - Document: {s['document_name']} | Chunk ID: {s['chunk_id']} | Score (Dist): {s['distance']:.2f}")

print("\n🔍 RETRIEVED CHUNK PREVIEW (First chunk):")
if result['retrieved_chunks']:
    print(result['retrieved_chunks'][0][:200] + "...")

## Section 4: Phase 3 - Context Window Study
We execute the automated evaluation study spanning dozens of questions across `512, 1024, 2048, 4096` token windows natively testing the limits of the hardware and RAG logic.

In [ ]:
from src.benchmark_runner import BenchmarkRunner
from IPython.display import display, Markdown, Image
import os
import shutil

# Ensure clean output directories
os.makedirs("results/dissertation_assets", exist_ok=True)

runner = BenchmarkRunner(rag, questions_path='data/evaluation_questions.json')
runner.run()
runner.generate_outputs()

# Copy to dissertation assets for packaging
for file in os.listdir("results"):
    if file.endswith(".png") or file.endswith(".csv") or file.endswith(".md"):
        shutil.copy(f"results/{file}", "results/dissertation_assets/")

print("\n✅ Phase 3 Complete! Analytics saved to disk.")

In [ ]:
display(Image(filename="results/dissertation_assets/comparison_dashboard.png"))

## Section 5: Results & Interpretation

In [ ]:
import pandas as pd
print("--- Context Window Summary ---")
df = pd.read_csv("results/dissertation_assets/context_window_summary.csv")
display(df)

print("\n--- Analysis Report ---")
display(Markdown(open("results/dissertation_assets/phase3_analysis.md").read()))

## Section 6: Final System Summary

In [ ]:
print("=========================================")
print("🏁 AUTOMOTIVE RAG SYSTEM - FINAL SUMMARY")
print("=========================================")
print(f"Documents Processed: {len(file_paths)}")
print(f"Chunks Created & Indexed: {len(all_chunks)}")
print(f"FAISS Index Size: {vs.index.ntotal} vectors")
print(f"Benchmark Questions Evaluated: {len(runner.questions)}")

best_window = df.loc[df['hit_rate'].idxmax()]['window_size'] if 'hit_rate' in df.columns else 'N/A'
print(f"\n🌟 RECOMMENDED CONTEXT WINDOW: {best_window} Tokens")
print("=========================================")

# Package dissertation assets into a zip for easy download in Colab
if IN_COLAB:
    !zip -r dissertation_assets.zip results/dissertation_assets/ > /dev/null
    print("\n📦 'dissertation_assets.zip' is ready for download in your Colab files panel!")